<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/CA2_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [28]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [29]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [30]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [31]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 5.43, 'feels_like': 0.43, 'temp_min': 5.43, 'temp_max': 5.43, 'humidity': 85, 'pressure': 1002, 'visibility': 10000, 'wind_speed': 9.02, 'wind_degree': 268, 'weather_main': 'Clouds', 'weather_desc': 'overcast clouds', 'cloud_coverage': 90, 'sunrise': 1774333095, 'sunset': 1774377864, 'fetched_at': '2026-03-24 21:48:55'}


In [33]:
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities
